# Snowflake lab setup with Azure OAuth

This notebook is a standalone lab guide for configuring Snowflake with Azure OAuth and generating the values needed for connector setup.

Reference setup documentation:
- Microsoft: https://learn.microsoft.com/en-us/connectors/snowflakev2/
- Snowflake: https://community.snowflake.com/s/article/Create-External-OAuth-Token-Using-Azure-AD-For-The-OAuth-Client-Itself

It automates as much as possible:
- validates Azure CLI sign-in
- creates or reuses the Entra resource app and client app
- adds the `ANALYST` app role
- creates service principals and grants the app role assignment
- requests a client credentials token
- decodes the JWT locally
- generates Snowflake SQL for the external OAuth integration
- optionally runs the Snowflake SQL if admin credentials are supplied

Manual steps still expected:
- sign up for the Snowflake lab or trial
- load sample data into Snowflake
- finish Copilot Studio connector setup

## Prerequisites

Before running the notebook:
1. Install Azure CLI and sign in with an account that can manage Entra app registrations.
2. Make sure Python can install packages from PyPI.
3. If you want the notebook to apply the Snowflake SQL directly, provide Snowflake admin connection details in the config cell.

In [ ]:
import importlib
import subprocess
import sys

required_packages = {
    'requests': 'requests',
    'jwt': 'PyJWT',
    'dotenv': 'python-dotenv',
    'snowflake.connector': 'snowflake-connector-python',
}

for module_name, package_name in required_packages.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

print('Required packages are available.')

## Configure the lab values

This section is only for values you already know before the one-time setup runs.

Known values from the setup instructions:
- app role display name: `ANALYST`
- app role value: `session:role:ANALYST`
- default security integration name: `EXTERNAL_OAUTH_AZURE`
- default Snowflake OAuth user: `SNOWSQL_OAUTH_USER`

Set if needed:
- `AZURE_TENANT_ID` only if you want to force a specific tenant; otherwise the notebook uses the current `az login` context
- `SNOWFLAKE_RESOURCE_APP_NAME` and `SNOWFLAKE_CLIENT_APP_NAME` if you want custom Entra app names
- `SNOWFLAKE_ACCOUNT`, `SNOWFLAKE_ADMIN_USER`, and `SNOWFLAKE_ADMIN_PASSWORD` if you want the notebook to run the Snowflake SQL directly
- `SNOWFLAKE_ACCOUNT` should be the exact Snowflake account identifier from the Snowflake URL or connection settings, not the username
- the notebook targets `SNOWFLAKE_SAMPLE_DATA` with schema `TPCH_SF1`

Optional overrides only for existing environments:
- `SNOWFLAKE_APP_ID_URI` if you are reusing an existing resource app URI
- `SNOWFLAKE_CLIENT_SECRET` if you already created a secret outside the notebook
- `SNOWFLAKE_PRINT_SQL=false` if you want the run section to skip printing the SQL text

Generated later by the Azure setup steps:
- `APP_ID_URI` when left blank
- `CLIENT_ID`
- a client secret kept hidden in notebook output
- token metadata and access tokens are regenerated each run and are not stored in `.env`

The Snowflake section later in the notebook uses the presence of the admin username and password in `.env` to decide whether to run directly in Snowflake or print the SQL for manual execution.

In [ ]:
import datetime as dt
import json
import os
from pathlib import Path
import textwrap
import uuid

import jwt
import requests
from dotenv import dotenv_values, find_dotenv, load_dotenv, set_key

existing_env_path = find_dotenv(usecwd=True)
ENV_FILE_PATH = Path(existing_env_path) if existing_env_path else Path.cwd() / '.env'

NOTEBOOK_ENV_KEYS = {
    'AZURE_TENANT_ID',
    'SNOWFLAKE_RESOURCE_APP_NAME',
    'SNOWFLAKE_CLIENT_APP_NAME',
    'SNOWFLAKE_APP_ID_URI',
    'SNOWFLAKE_CLIENT_SECRET',
    'SNOWFLAKE_RESOURCE_APP_ID',
    'SNOWFLAKE_CLIENT_ID',
    'SNOWFLAKE_TOKEN_ENDPOINT',
    'SNOWFLAKE_JWKS_URI',
    'SNOWFLAKE_ISSUER',
    'SNOWFLAKE_ACCOUNT',
    'SNOWFLAKE_ADMIN_USER',
    'SNOWFLAKE_ADMIN_PASSWORD',
    'SNOWFLAKE_ADMIN_ROLE',
    'SNOWFLAKE_WAREHOUSE',
    'SNOWFLAKE_DATABASE',
    'SNOWFLAKE_SCHEMA',
    'SNOWFLAKE_SECURITY_INTEGRATION',
    'SNOWFLAKE_OAUTH_USER',
    'SNOWFLAKE_PRINT_SQL',
}

def reload_env_file():
    ENV_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)

    if ENV_FILE_PATH.exists():
        current_values = dotenv_values(ENV_FILE_PATH)
        for key in NOTEBOOK_ENV_KEYS:
            value = current_values.get(key)
            if value is None or str(value).strip() == '':
                os.environ.pop(key, None)
            else:
                os.environ[key] = str(value).strip()

    load_dotenv(dotenv_path=ENV_FILE_PATH, override=True)
    return ENV_FILE_PATH

reload_env_file()

# Known values from the setup instructions.
APP_ROLE_DISPLAY_NAME = 'ANALYST'
APP_ROLE_VALUE = 'session:role:ANALYST'
SECURITY_INTEGRATION_NAME = os.getenv('SNOWFLAKE_SECURITY_INTEGRATION', globals().get('SECURITY_INTEGRATION_NAME', 'EXTERNAL_OAUTH_AZURE')).strip()
SNOWFLAKE_OAUTH_USER = os.getenv('SNOWFLAKE_OAUTH_USER', globals().get('SNOWFLAKE_OAUTH_USER', 'SNOWSQL_OAUTH_USER')).strip()

# Inputs you may know before running the one-time setup.
TENANT_ID = os.getenv('AZURE_TENANT_ID', globals().get('TENANT_ID', '')).strip()
RESOURCE_APP_NAME = os.getenv('SNOWFLAKE_RESOURCE_APP_NAME', globals().get('RESOURCE_APP_NAME', 'Snowflake OAuth Resource')).strip()
CLIENT_APP_NAME = os.getenv('SNOWFLAKE_CLIENT_APP_NAME', globals().get('CLIENT_APP_NAME', 'Snowflake OAuth Client')).strip()

# Optional overrides only if you are reusing existing Azure settings.
APP_ID_URI = os.getenv('SNOWFLAKE_APP_ID_URI', globals().get('APP_ID_URI', '')).strip()
CLIENT_SECRET = os.getenv('SNOWFLAKE_CLIENT_SECRET', globals().get('CLIENT_SECRET', '')).strip()
RESOURCE_APP_ID = os.getenv('SNOWFLAKE_RESOURCE_APP_ID', globals().get('RESOURCE_APP_ID', '')).strip()
CLIENT_ID = os.getenv('SNOWFLAKE_CLIENT_ID', globals().get('CLIENT_ID', '')).strip()
TOKEN_ENDPOINT = os.getenv('SNOWFLAKE_TOKEN_ENDPOINT', globals().get('TOKEN_ENDPOINT', '')).strip()
JWKS_URI = os.getenv('SNOWFLAKE_JWKS_URI', globals().get('JWKS_URI', '')).strip()
ISSUER = os.getenv('SNOWFLAKE_ISSUER', globals().get('ISSUER', '')).strip()
GENERATE_NEW_CLIENT_SECRET = not bool(CLIENT_SECRET)

def normalize_snowflake_account(value):
    account = (value or '').strip()
    if account.startswith('https://'):
        account = account[len('https://'):]
    elif account.startswith('http://'):
        account = account[len('http://'):]

    account = account.rstrip('/')
    if '.snowflakecomputing.com' in account:
        account = account.split('.snowflakecomputing.com', 1)[0]

    return account

# If admin credentials are present in .env, the notebook will run the Snowflake SQL directly.
# Otherwise, the same section will print the SQL for manual execution.
SNOWFLAKE_ACCOUNT = normalize_snowflake_account(os.getenv('SNOWFLAKE_ACCOUNT', globals().get('SNOWFLAKE_ACCOUNT', '')))
SNOWFLAKE_ADMIN_USER = os.getenv('SNOWFLAKE_ADMIN_USER', globals().get('SNOWFLAKE_ADMIN_USER', '')).strip()
SNOWFLAKE_ADMIN_PASSWORD = os.getenv('SNOWFLAKE_ADMIN_PASSWORD', globals().get('SNOWFLAKE_ADMIN_PASSWORD', '')).strip()
SNOWFLAKE_ADMIN_ROLE = os.getenv('SNOWFLAKE_ADMIN_ROLE', globals().get('SNOWFLAKE_ADMIN_ROLE', 'ACCOUNTADMIN')).strip()
SNOWFLAKE_WAREHOUSE = os.getenv('SNOWFLAKE_WAREHOUSE', globals().get('SNOWFLAKE_WAREHOUSE', 'COMPUTE_WH')).strip()
SNOWFLAKE_DATABASE = os.getenv('SNOWFLAKE_DATABASE', globals().get('SNOWFLAKE_DATABASE', 'SNOWFLAKE_SAMPLE_DATA')).strip()
if SNOWFLAKE_DATABASE.lower() == 'adventureworks':
    SNOWFLAKE_DATABASE = 'SNOWFLAKE_SAMPLE_DATA'

SNOWFLAKE_SCHEMA = os.getenv('SNOWFLAKE_SCHEMA', globals().get('SNOWFLAKE_SCHEMA', 'TPCH_SF1')).strip()
if SNOWFLAKE_DATABASE.upper() == 'SNOWFLAKE_SAMPLE_DATA' and SNOWFLAKE_SCHEMA.lower() == 'public':
    SNOWFLAKE_SCHEMA = 'TPCH_SF1'

PRINT_SNOWFLAKE_SQL = os.getenv('SNOWFLAKE_PRINT_SQL', str(globals().get('PRINT_SNOWFLAKE_SQL', 'true'))).strip().lower() not in {'0', 'false', 'no'}
RUN_SNOWFLAKE_CHANGES = bool(SNOWFLAKE_ADMIN_USER and SNOWFLAKE_ADMIN_PASSWORD)

print(f'Lab variables loaded successfully. Reloaded values from .env file: {ENV_FILE_PATH}')
print(f'Snowflake target: {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA}')
print(f"Snowflake execution mode: {'direct apply' if RUN_SNOWFLAKE_CHANGES else 'print SQL only'}")

## Load Azure helper functions

The next code cell defines reusable helpers for:
- checking Azure CLI sign-in
- calling Microsoft Graph
- creating or reusing app registrations and service principals
- assigning the Snowflake `ANALYST` app role
- generating a client secret when needed

These helpers are used by the later execution cells so the lab can stay mostly automated.

In [ ]:
GRAPH_BASE = 'https://graph.microsoft.com/v1.0'
AZ_COMMAND = 'az.cmd' if os.name == 'nt' else 'az'

def persist_env_values(values, reload_after_save=True):
    ENV_FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not ENV_FILE_PATH.exists():
        ENV_FILE_PATH.touch()

    persisted_keys = []
    for key, value in values.items():
        if value is None or value == '':
            continue
        if not isinstance(value, str):
            value = str(value)
        set_key(str(ENV_FILE_PATH), key, value)
        persisted_keys.append(key)

    if reload_after_save:
        reload_env_file()

    return persisted_keys

def run_command(args):
    command = list(args)
    if command and command[0] == 'az':
        command[0] = AZ_COMMAND

    try:
        result = subprocess.run(command, capture_output=True, text=True)
    except FileNotFoundError as exc:
        raise RuntimeError(
            'Azure CLI is not available to the notebook kernel. On Windows, the notebook uses az.cmd. If Azure CLI is already installed, restart the kernel or VS Code and try again.'
        ) from exc

    if result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(args)}\n{result.stderr.strip()}")
    return result.stdout.strip()

def ensure_az_login():
    try:
        account = json.loads(run_command(['az', 'account', 'show', '-o', 'json']))
    except Exception as exc:
        raise RuntimeError('Azure CLI is installed but not logged in for this context. Run az login in a terminal and rerun the notebook.') from exc

    if TENANT_ID and account.get('tenantId') != TENANT_ID:
        print(f"Current Azure CLI tenant is {account.get('tenantId')} but TENANT_ID is set to {TENANT_ID}.")
    return account

def get_graph_headers():
    token = run_command(['az', 'account', 'get-access-token', '--resource-type', 'ms-graph', '--query', 'accessToken', '-o', 'tsv'])
    return {
        'Authorization': f'Bearer {token}',
        'Content-Type': 'application/json',
    }

def graph_request(method, url, payload=None):
    if url.startswith('http'):
        request_url = url
    else:
        request_url = f'{GRAPH_BASE}{url}'

    response = requests.request(
        method=method,
        url=request_url,
        headers=get_graph_headers(),
        json=payload,
        timeout=60,
    )

    if response.status_code >= 400:
        raise RuntimeError(f'{method} {request_url} failed: {response.status_code} {response.text}')

    if response.status_code == 204 or not response.text.strip():
        return {}

    return response.json()

def graph_collection(url):
    items = []
    next_url = url
    while next_url:
        data = graph_request('GET', next_url)
        items.extend(data.get('value', []))
        next_url = data.get('@odata.nextLink')
    return items

def escape_filter_value(value):
    return value.replace("'", "''")

def ensure_application(display_name):
    apps = graph_collection(f"/applications?$filter=displayName eq '{escape_filter_value(display_name)}'")
    if apps:
        return apps[0], False

    payload = {
        'displayName': display_name,
        'signInAudience': 'AzureADMyOrg',
    }
    return graph_request('POST', '/applications', payload=payload), True

def ensure_service_principal(app_id):
    sps = graph_collection(f"/servicePrincipals?$filter=appId eq '{app_id}'")
    if sps:
        return sps[0], False

    return graph_request('POST', '/servicePrincipals', payload={'appId': app_id}), True

def ensure_resource_app_settings(app, app_id_uri, role_display_name, role_value):
    app_roles = list(app.get('appRoles') or [])
    target_role = next((role for role in app_roles if role.get('value') == role_value), None)

    if not target_role:
        target_role = {
            'allowedMemberTypes': ['Application'],
            'description': f'Scope permission for {role_display_name} Snowflake Role',
            'displayName': role_display_name,
            'id': str(uuid.uuid4()),
            'isEnabled': True,
            'value': role_value,
        }
        app_roles.append(target_role)

    payload = {
        'identifierUris': [app_id_uri],
        'api': {
            'requestedAccessTokenVersion': 2,
        },
        'appRoles': app_roles,
    }

    graph_request('PATCH', f"/applications/{app['id']}", payload=payload)
    refreshed = graph_request('GET', f"/applications/{app['id']}")
    target_role = next(role for role in refreshed.get('appRoles', []) if role.get('value') == role_value)
    return refreshed, target_role

def ensure_client_permissions(app, resource_app_id, app_role_id):
    required_resource_access = list(app.get('requiredResourceAccess') or [])
    target_entry = next((entry for entry in required_resource_access if entry.get('resourceAppId') == resource_app_id), None)

    if not target_entry:
        target_entry = {
            'resourceAppId': resource_app_id,
            'resourceAccess': [],
        }
        required_resource_access.append(target_entry)

    if not any(item.get('id') == app_role_id for item in target_entry.get('resourceAccess', [])):
        target_entry.setdefault('resourceAccess', []).append({
            'id': app_role_id,
            'type': 'Role',
        })

    graph_request('PATCH', f"/applications/{app['id']}", payload={
        'requiredResourceAccess': required_resource_access,
    })
    return graph_request('GET', f"/applications/{app['id']}")

def ensure_app_role_assignment(client_sp_id, resource_sp_id, app_role_id):
    assignments = graph_collection(f"/servicePrincipals/{client_sp_id}/appRoleAssignments")
    for assignment in assignments:
        if assignment.get('resourceId') == resource_sp_id and assignment.get('appRoleId') == app_role_id:
            return assignment, False

    payload = {
        'principalId': client_sp_id,
        'resourceId': resource_sp_id,
        'appRoleId': app_role_id,
    }
    return graph_request('POST', f"/servicePrincipals/{client_sp_id}/appRoleAssignments", payload=payload), True

def add_client_secret(app_object_id, display_name='snowflake-lab-secret', valid_days=365):
    end_time = (dt.datetime.now(dt.timezone.utc) + dt.timedelta(days=valid_days)).strftime('%Y-%m-%dT%H:%M:%SZ')
    payload = {
        'passwordCredential': {
            'displayName': display_name,
            'endDateTime': end_time,
        }
    }
    result = graph_request('POST', f"/applications/{app_object_id}/addPassword", payload=payload)
    return result['secretText'].strip()

print('Helper functions loaded.')

## Create or reuse the Entra apps

This step maps to the Azure app registration work in the setup instructions.

It will:
- create or reuse the resource app registration that acts as the OAuth server
- create the `ANALYST` app role with value `session:role:ANALYST`
- create or reuse the client app registration
- grant the client app permission to the resource app
- discover the OpenID metadata values used later in Snowflake
- generate a client secret only if you did not provide one already

The client secret is kept in memory for the rest of the notebook session and is never printed to notebook output.

In [ ]:
account = ensure_az_login()
tenant_id = TENANT_ID or account['tenantId']

resource_app, resource_created = ensure_application(RESOURCE_APP_NAME)
app_id_uri = APP_ID_URI or f"api://{resource_app['appId']}"
resource_app, analyst_role = ensure_resource_app_settings(
    resource_app,
    app_id_uri,
    APP_ROLE_DISPLAY_NAME,
    APP_ROLE_VALUE,
)

client_app, client_created = ensure_application(CLIENT_APP_NAME)
client_app = ensure_client_permissions(client_app, resource_app['appId'], analyst_role['id'])

resource_sp, resource_sp_created = ensure_service_principal(resource_app['appId'])
client_sp, client_sp_created = ensure_service_principal(client_app['appId'])
assignment, assignment_created = ensure_app_role_assignment(client_sp['id'], resource_sp['id'], analyst_role['id'])

client_secret = CLIENT_SECRET
if GENERATE_NEW_CLIENT_SECRET:
    client_secret = add_client_secret(client_app['id'])
    client_secret_status = 'generated in memory for this notebook session'
else:
    client_secret_status = 'loaded from environment and kept hidden'

CLIENT_SECRET = client_secret

openid_config = requests.get(
    f"https://login.microsoftonline.com/{tenant_id}/v2.0/.well-known/openid-configuration",
    timeout=60,
).json()

TOKEN_ENDPOINT = openid_config['token_endpoint']
JWKS_URI = openid_config['jwks_uri']
ISSUER = openid_config['issuer']
CLIENT_ID = client_app['appId']
RESOURCE_APP_ID = resource_app['appId']

summary = {
    'tenant_id': tenant_id,
    'resource_app_name': RESOURCE_APP_NAME,
    'resource_app_id': RESOURCE_APP_ID,
    'app_role_display_name': APP_ROLE_DISPLAY_NAME,
    'app_role_value': APP_ROLE_VALUE,
    'app_id_uri': app_id_uri,
    'client_app_name': CLIENT_APP_NAME,
    'client_id': CLIENT_ID,
    'client_secret_status': client_secret_status,
    'token_endpoint': TOKEN_ENDPOINT,
    'jwks_uri': JWKS_URI,
    'issuer': ISSUER,
    'resource_app_created': resource_created,
    'client_app_created': client_created,
    'role_assignment_created': assignment_created,
    'env_file_target': str(ENV_FILE_PATH),
}

print(json.dumps(summary, indent=2))

## Request the OAuth token and inspect the claims

This cell replaces the manual token request and JWT inspection steps.

If the notebook just created a new client secret, Entra ID can take a short time to accept it. The code below retries automatically before failing.

In [ ]:
import time

max_attempts = 6 if GENERATE_NEW_CLIENT_SECRET else 1
last_response = None

for attempt in range(1, max_attempts + 1):
    token_response = requests.post(
        TOKEN_ENDPOINT,
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
        data={
            'client_id': CLIENT_ID,
            'client_secret': CLIENT_SECRET,
            'grant_type': 'client_credentials',
            'scope': f'{app_id_uri}/.default',
        },
        timeout=60,
    )

    last_response = token_response
    if token_response.ok:
        break

    response_text = token_response.text
    is_new_secret_delay = (
        GENERATE_NEW_CLIENT_SECRET
        and token_response.status_code == 401
        and 'AADSTS7000215' in response_text
        and attempt < max_attempts
    )

    if is_new_secret_delay:
        print(f'New client secret may still be propagating in Entra ID. Retry {attempt} of {max_attempts}...')
        time.sleep(10)
        continue

    raise RuntimeError(f'Token request failed: {token_response.status_code} {response_text}')
else:
    status_code = last_response.status_code if last_response is not None else 'unknown'
    response_text = last_response.text if last_response is not None else 'no response'
    raise RuntimeError(f'Token request failed after {max_attempts} attempts: {status_code} {response_text}')

token_payload = token_response.json()
access_token = token_payload['access_token']
claims = jwt.decode(access_token, options={'verify_signature': False, 'verify_aud': False, 'verify_exp': False})
token_audience = claims.get('aud', '')

important_claims = {
    'iss': claims.get('iss'),
    'aud': token_audience,
    'roles': claims.get('roles'),
    'appid': claims.get('appid'),
    'sub': claims.get('sub'),
    'tid': claims.get('tid'),
    'env_file_target': str(ENV_FILE_PATH),
}

print(json.dumps(important_claims, indent=2))

## Save the generated Azure values

The next code cell writes the stable, reusable values to the workspace `.env` file.

Ephemeral values such as access tokens, token metadata, and other discovery results are intentionally not persisted because they can be regenerated when needed.

In [ ]:
# Persist stable, reusable values to the workspace .env file.

persisted_keys = persist_env_values({
    'AZURE_TENANT_ID': tenant_id if 'tenant_id' in globals() else TENANT_ID,
    'SNOWFLAKE_RESOURCE_APP_NAME': RESOURCE_APP_NAME,
    'SNOWFLAKE_CLIENT_APP_NAME': CLIENT_APP_NAME,
    'SNOWFLAKE_APP_ID_URI': app_id_uri if 'app_id_uri' in globals() else APP_ID_URI,
    'SNOWFLAKE_CLIENT_ID': CLIENT_ID,
    'SNOWFLAKE_CLIENT_SECRET': CLIENT_SECRET,
    'SNOWFLAKE_DATABASE': SNOWFLAKE_DATABASE,
    'SNOWFLAKE_SCHEMA': SNOWFLAKE_SCHEMA,
})

print(json.dumps({
    'env_file': str(ENV_FILE_PATH),
    'persisted_keys': persisted_keys,
}, indent=2))

## Generate or run the Snowflake SQL

This section builds the Snowflake SQL using the Azure values gathered above.

If `SNOWFLAKE_ADMIN_USER` and `SNOWFLAKE_ADMIN_PASSWORD` are present in `.env`, the same code cell connects to Snowflake and applies the statements directly.

If they are not present, the cell prints the SQL for manual execution. You can also keep SQL printing enabled during direct execution with `SNOWFLAKE_PRINT_SQL=true`.

In [ ]:
oauth_login_name = claims.get('sub', '')

audience_values = []
for value in [claims.get('aud', ''), app_id_uri]:
    if value and value not in audience_values:
        audience_values.append(value)

audience_list_sql = ', '.join(f"'{value}'" for value in audience_values)
is_sample_data_database = SNOWFLAKE_DATABASE.upper() == 'SNOWFLAKE_SAMPLE_DATA'

if is_sample_data_database:
    database_grants_sql = textwrap.dedent(f"""
    GRANT IMPORTED PRIVILEGES ON DATABASE {SNOWFLAKE_DATABASE} TO ROLE {APP_ROLE_DISPLAY_NAME};
    """).strip()
else:
    database_grants_sql = textwrap.dedent(f"""
    GRANT USAGE ON DATABASE {SNOWFLAKE_DATABASE} TO ROLE {APP_ROLE_DISPLAY_NAME};
    GRANT USAGE ON SCHEMA {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA} TO ROLE {APP_ROLE_DISPLAY_NAME};
    GRANT SELECT ON ALL TABLES IN SCHEMA {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA} TO ROLE {APP_ROLE_DISPLAY_NAME};
    GRANT SELECT ON FUTURE TABLES IN SCHEMA {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA} TO ROLE {APP_ROLE_DISPLAY_NAME};
    """).strip()

security_integration_sql = textwrap.dedent(f"""
CREATE OR REPLACE SECURITY INTEGRATION {SECURITY_INTEGRATION_NAME}
TYPE = EXTERNAL_OAUTH
ENABLED = TRUE
EXTERNAL_OAUTH_TYPE = AZURE
EXTERNAL_OAUTH_ISSUER = '{ISSUER}'
EXTERNAL_OAUTH_JWS_KEYS_URL = '{JWKS_URI}'
EXTERNAL_OAUTH_AUDIENCE_LIST = ({audience_list_sql})
EXTERNAL_OAUTH_TOKEN_USER_MAPPING_CLAIM = 'sub'
EXTERNAL_OAUTH_SNOWFLAKE_USER_MAPPING_ATTRIBUTE = 'LOGIN_NAME';
""").strip()

token_validation_sql = textwrap.dedent(f"""
SELECT SYSTEM$VERIFY_EXTERNAL_OAUTH_TOKEN('{access_token}');
""").strip()

provisioning_sql = textwrap.dedent(f"""
CREATE ROLE IF NOT EXISTS {APP_ROLE_DISPLAY_NAME};

CREATE USER IF NOT EXISTS {SNOWFLAKE_OAUTH_USER}
LOGIN_NAME = '{oauth_login_name}'
DISPLAY_NAME = 'SnowSQL OAuth User'
DEFAULT_ROLE = {APP_ROLE_DISPLAY_NAME}
DEFAULT_WAREHOUSE = {SNOWFLAKE_WAREHOUSE}
COMMENT = 'A system user for OAuth based connectivity';

ALTER USER IF EXISTS {SNOWFLAKE_OAUTH_USER}
SET DEFAULT_ROLE = {APP_ROLE_DISPLAY_NAME}
DEFAULT_WAREHOUSE = {SNOWFLAKE_WAREHOUSE};

GRANT ROLE {APP_ROLE_DISPLAY_NAME} TO USER {SNOWFLAKE_OAUTH_USER};
GRANT USAGE ON WAREHOUSE {SNOWFLAKE_WAREHOUSE} TO ROLE {APP_ROLE_DISPLAY_NAME};
{database_grants_sql}
""").strip()

statements = [
    security_integration_sql,
    provisioning_sql,
    token_validation_sql,
]

print(f"Snowflake execution mode: {'direct apply' if RUN_SNOWFLAKE_CHANGES else 'print SQL only'}")
print('Audience values used by Snowflake integration:')
print(json.dumps(audience_values, indent=2))
print()

if PRINT_SNOWFLAKE_SQL or not RUN_SNOWFLAKE_CHANGES:
    print('--- Security integration SQL ---')
    print(security_integration_sql)
    print()
    print('--- Token verification SQL ---')
    print(token_validation_sql)
    print()
    print('--- User and grants SQL ---')
    print(provisioning_sql)

if RUN_SNOWFLAKE_CHANGES:
    if not SNOWFLAKE_ACCOUNT:
        raise ValueError('SNOWFLAKE_ACCOUNT must be set in .env when Snowflake admin credentials are provided.')

    import snowflake.connector
    from snowflake.connector.errors import Error as SnowflakeError

    try:
        connection = snowflake.connector.connect(
            account=SNOWFLAKE_ACCOUNT,
            user=SNOWFLAKE_ADMIN_USER,
            password=SNOWFLAKE_ADMIN_PASSWORD,
            role=SNOWFLAKE_ADMIN_ROLE,
            warehouse=SNOWFLAKE_WAREHOUSE,
            database=SNOWFLAKE_DATABASE,
            schema=SNOWFLAKE_SCHEMA,
        )
    except SnowflakeError as exc:
        message = str(exc)
        if '404 Not Found' in message and 'login-request' in message:
            raise ValueError(
                f"Snowflake connection failed because SNOWFLAKE_ACCOUNT='{SNOWFLAKE_ACCOUNT}' does not appear to be a valid account identifier. Use the exact account identifier from the Snowflake URL or connection settings, not the username."
            ) from exc
        raise

    try:
        with connection.cursor() as cursor:
            for statement in statements:
                for chunk in [part.strip() for part in statement.split(';') if part.strip()]:
                    print(f'Running: {chunk[:100]}...')
                    cursor.execute(chunk)
                    try:
                        rows = cursor.fetchall()
                        if rows:
                            print(rows)
                    except Exception:
                        pass
    finally:
        connection.close()
else:
    print('Snowflake admin credentials were not found in .env. Review the SQL above and run it manually in Snowflake.')

## Values to carry into Copilot Studio

After the earlier setup cells run, use the following values when you configure the connector:
- Client ID: `CLIENT_ID`
- Token endpoint: `TOKEN_ENDPOINT`
- Scope: `f'{app_id_uri}/.default'`
- Audience: `app_id_uri`

Only the stable values are saved to `.env`. Dynamic values such as the token endpoint, issuer metadata, and access token are regenerated when the notebook runs.

## Test the Snowflake OAuth connection

This optional section uses the OAuth access token generated earlier in the notebook and attempts a real Snowflake connection.

It confirms:
- the OAuth token can authenticate successfully
- the Snowflake user mapping is working
- the configured role, warehouse, database, and schema are available

If you open the notebook in a fresh kernel, rerun the earlier setup and token sections before running this test.

In [ ]:
# Test the Snowflake OAuth connection using the access token from the earlier sections.

reload_env_file()

if 'access_token' not in globals() or not access_token:
    raise ValueError('Run the OAuth token section earlier in the notebook before running this test.')

if not SNOWFLAKE_ACCOUNT:
    raise ValueError('SNOWFLAKE_ACCOUNT must be set in .env before running the OAuth connection test.')

import snowflake.connector

oauth_connector_user = ''
if 'claims' in globals() and isinstance(claims, dict):
    oauth_connector_user = (claims.get('sub') or '').strip()

if not oauth_connector_user:
    decoded_claims = jwt.decode(access_token, options={'verify_signature': False, 'verify_aud': False, 'verify_exp': False})
    oauth_connector_user = (decoded_claims.get('sub') or '').strip()

if not oauth_connector_user:
    oauth_connector_user = SNOWFLAKE_OAUTH_USER

test_query = textwrap.dedent("""
SELECT
    CURRENT_USER() AS current_user,
    CURRENT_ROLE() AS current_role,
    CURRENT_WAREHOUSE() AS current_warehouse,
    CURRENT_DATABASE() AS current_database,
    CURRENT_SCHEMA() AS current_schema
""").strip()

oauth_results = {}

with snowflake.connector.connect(
    account=SNOWFLAKE_ACCOUNT,
    user=oauth_connector_user,
    authenticator='oauth',
    token=access_token,
    role=APP_ROLE_DISPLAY_NAME,
    warehouse=SNOWFLAKE_WAREHOUSE,
    database=SNOWFLAKE_DATABASE,
    schema=SNOWFLAKE_SCHEMA,
) as oauth_connection:
    with oauth_connection.cursor() as cursor:
        cursor.execute(test_query)
        session_row = cursor.fetchone()
        session_columns = [column[0].lower() for column in cursor.description]
        session_info = dict(zip(session_columns, session_row))

        oauth_results = {
            'connection_test': 'passed',
            'connector_user_parameter': oauth_connector_user,
            **session_info,
        }

        if SNOWFLAKE_DATABASE.upper() == 'SNOWFLAKE_SAMPLE_DATA' and SNOWFLAKE_SCHEMA.upper() == 'TPCH_SF1':
            cursor.execute(f'SELECT COUNT(*) AS row_count FROM {SNOWFLAKE_DATABASE}.{SNOWFLAKE_SCHEMA}.REGION')
            region_row = cursor.fetchone()
            oauth_results['sample_region_row_count'] = region_row[0]

print(json.dumps(oauth_results, indent=2))